# 1. Getting started: from a PED entry to a SOURSOP trajectory

The [Protein Ensemble Database (PED)](https://proteinensemble.org) is the main public repository of conformational ensembles of intrinsically disordered proteins. `soursop.ped` lets you search PED, look at its entries, and load an ensemble straight into SOURSOP as an `SSTrajectory`, with one frame per conformer.

This notebook walks through the basic loop with Sic1 (entry `PED00001`, three ensembles of the disordered N-terminal region of phosphorylated Sic1):

1. find entries with a search,
2. look at an entry's metadata and its ensembles,
3. load an ensemble,
4. analyse it with SOURSOP, and check the results against PED's own numbers.

**Identifiers.** A PED *entry* (`PED00001`) groups one or more *ensembles* deposited together; an ensemble is written `PED00001e001`. Functions that need an ensemble accept the full form, or the entry plus `ensemble_id`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from soursop import ped

# Figure style: Arial throughout, text kept editable in Illustrator, no grid
# lines, and the xmgrace colour order (black, red, blue, green, purple)
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.grid": False,
    "axes.prop_cycle": plt.cycler(color=["black", "red", "blue", "green", "purple"]),
})

# Downloaded ensembles are kept here and reused, so each is only fetched once
# (shared by all the notebooks in this folder)
CACHE = "ped_cache"

## Find entries

`ped.search()` returns identifiers. With a plain string it runs PED's free-text search.

In [ ]:
hits = ped.search("sic1")
print(f"{len(hits)} entries match 'sic1':")
print(hits)

## Look at an entry

`ped.get_entry()` returns a `PEDEntry`: the metadata you usually want as attributes, with the full JSON record from PED in `.raw`.

In [ ]:
entry = ped.get_entry("PED00001")
print(entry)
print()
print("Title:       ", entry.title)
print("Authors:     ", ", ".join(entry.authors))
print("UniProt:     ", entry.uniprot_accessions)
print("Publication: ", entry.publication)
print("Methods:     ", ", ".join(entry.ontology_terms))
print("Cross-refs:  ", entry.cross_references)

Each ensemble in the entry is a `PEDEnsembleSummary`, with its size, whether it is C-alpha only (coarse-grained), and the chain sequence(s).

In [ ]:
for ens in entry.ensembles:
    chain = ens.chains[0]
    print(f"{ens.identifier}: {ens.n_models} conformers, C-alpha only = {ens.only_ca}, "
          f"chain {chain.chain_name} ({len(chain.sequence)} residues)")
print()
print("Sequence:", entry.ensembles[0].chains[0].sequence)

## Load an ensemble

`ped.load_ensemble()` downloads the ensemble's multi-model PDB file and reads it into an `SSTrajectory`. With `cache_dir` set, the file is kept and reused on later calls; without it, the download goes to a temporary directory that is deleted once the ensemble is read in.

In [ ]:
traj = ped.load_ensemble("PED00001e001", cache_dir=CACHE)
print(traj)

protein = traj.proteinTrajectoryList[0]
print(protein.n_residues, "residues")

From here it is an ordinary SOURSOP object: every `SSProtein` method works as it would on a simulation you ran yourself.

In [ ]:
rg = protein.get_radius_of_gyration()                  # per conformer, Angstroms
re = protein.get_end_to_end_distance(mode="CA")         # per conformer, Angstroms
asph = protein.get_asphericity(verbose=False)

print(f"<Rg>          = {rg.mean():.1f} +/- {rg.std():.1f} A")
print(f"<Re>          = {re.mean():.1f} +/- {re.std():.1f} A")
print(f"<asphericity> = {asph.mean():.2f}")

## Check against PED's own numbers

PED analyses every ensemble itself, and `load_ensemble` gives you the raw coordinates, so the two can be compared directly. PED's summary is in `ped_statistics`. Note that PED reports distances in **nanometres**, whereas SOURSOP uses Angstroms.

In [ ]:
stats = entry.ensemble("e001").ped_statistics
print("PED summary fields:", ", ".join(sorted(stats)))
print()
print(f"{'':10s}{'<Rg> (A)':>10s}{'sd(Rg) (A)':>12s}{'<Re> (A)':>10s}")
print(f"{'PED':10s}{10*stats['rg_mean']:10.2f}{10*stats['rg_std']:12.2f}{10*stats['end_to_end_mean']:10.2f}")
print(f"{'SOURSOP':10s}{rg.mean():10.2f}{rg.std():12.2f}{re.mean():10.2f}")

The two agree to the second decimal place: an independent check that the ensemble arrived intact and that both tools measure the same thing (all-atom Rg, and the C-alpha end-to-end distance).

## Compare the entry's ensembles

`PEDEnsembleSummary.load()` loads an individual ensemble. Here we compare the Rg distributions of the three Sic1 ensembles. With only 10-11 conformers each, a cumulative distribution shows the data more honestly than a histogram.

In [ ]:
fig, ax = plt.subplots(figsize=(3.4, 2.8))
for ens in entry.ensembles:
    p = ens.load(cache_dir=CACHE).proteinTrajectoryList[0]
    values = np.sort(p.get_radius_of_gyration())
    fraction = np.arange(1, len(values) + 1) / len(values)
    ax.step(values, fraction, where="post", label=ens.identifier)

ax.set_xlabel("Radius of gyration (Å)")
ax.set_ylabel("Cumulative fraction of conformers")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Next steps

* `02_searching_ped` - the different ways to search PED.
* `03_one_protein_many_methods` - comparing ensembles of the same protein from different methods.
* `07_weighted_ensembles_and_downloads` - weighted ensembles, and saving files to disk.